In [0]:
ROOT = "/Workspace/Users/melaniepieterse1@gmail.com/Loan_Offer_Model"
CATALOG_PATH = f"{ROOT}/conf/base/catalog.yml"


In [0]:
import sys
sys.path.insert(0, ROOT)

from src.catalog_loader import Catalog
catalog = Catalog(spark, CATALOG_PATH)


In [0]:
customers_df = catalog.load("customer")
loans_df = catalog.load("loans")
transactions_df = catalog.load("transactions")

print(f"customers_df: {customers_df.count()} rows")
print(f"loans_df: {loans_df.count()} rows")
print(f"transactions_df: {transactions_df.count()} rows")


In [0]:
display(customers_df)
display(loans_df)
display(transactions_df)

Databricks filtered table. Run in Databricks to view.

In [0]:
%pip install ydata-profiling visions tzdata
import pandas as pd
from ydata_profiling import ProfileReport

In [0]:
# Evaluate the customer data profiles, identify outliers or missing values.
df_profile = ProfileReport(customers_df.toPandas(), minimal=True, title="Profiling Report", progress_bar=False, infer_dtypes=False)
profile_html = df_profile.to_html()
displayHTML(profile_html)

df_profile = ProfileReport(loans_df.toPandas(), minimal=True, title="Profiling Report", progress_bar=False, infer_dtypes=False)
profile_html = df_profile.to_html()
displayHTML(profile_html)

df_profile = ProfileReport(transactions_df.toPandas(), minimal=True, title="Profiling Report", progress_bar=False, infer_dtypes=False)
profile_html = df_profile.to_html()
displayHTML(profile_html)

# Customer Data 
- Explore transactional data as possible indication of best INCOME impute method for misssing INCOME data
- Explore options to either infer the correct  age for unrealitic ages using gender and income, else if not confident with this imputation approach - exclude.

In [0]:
from pyspark.sql.functions import *

# Use transactions data to identify potential income data for customers with missing income data. The income field is the monthly income data for each customer.

# Get the customer ID's that have missing income data. There are over 1,324 customers with missing income. 
monthly_income_to_inpute_df = customers_df.filter(col('INCOME').isNull()).select('ID').join(transactions_df.filter(col('AMOUNT') > 0), on = 'ID', how = 'left')
display(monthly_income_to_inpute_df)

# Use the transactional data to determine if there are a consistent income stream per customer ID  (postive transaction amounts) by looking at the statistics and any differences between the mode and max amounts
consistent_income = monthly_income_to_inpute_df.groupBy('ID').agg(count('*'), median(col('AMOUNT')).alias('median_AMOUNT'), avg(col('AMOUNT')).alias('average_AMOUNT'),mode(col('AMOUNT')).alias('mode_AMOUNT'),max(col('AMOUNT')).alias('max_AMOUNT')).withColumn('ADDITIONAL_INCOME', col('max_AMOUNT') - col('mode_AMOUNT'))
display(consistent_income)

# There are 344 customers  that show consistent income. This was deterimed if there were no difference between the mode and median amount, and if there were more than one income transaction.
inpute_consistent_income = consistent_income.filter(col('ADDITIONAL_INCOME')==0).select('ID','median_AMOUNT').withColumnRenamed('median_AMOUNT','INCOME')
display(inpute_consistent_income)

# Apply a filter to work with  the remaining customers with missing income data
filter_ids = [row.ID for row in inpute_consistent_income.select('ID').collect()]
monthly_income_to_inpute_df = monthly_income_to_inpute_df.filter(~col('ID').isin(filter_ids))
display(monthly_income_to_inpute_df)


#
additional_income = consistent_income.filter(col('ADDITIONAL_INCOME')!=0)
display(additional_income)


# Calculate how often the mode occurs for customers with additional income (ADDITIONAL_INCOME > 0)
def calculate_mode_frequency(additional_income_df, transactions_df):
    """
    For each customer in additional_income_df, compute how frequently the
    mode (most common) positive transaction amount appears in transactions_df.
    Also compute the count of distinct AMOUNT values and the overall count(*) of
    positive transactions for comparison.
    Returns a DataFrame with ID, mode_AMOUNT, mode_count, distinct_AMOUNT_count,
    and total_count.
    """
    # Get the IDs of customers with additional income
    income_ids = additional_income_df.filter(col('ADDITIONAL_INCOME') > 0).select('ID')

    # Join with positive transactions for those customers
    positive_txns = transactions_df.filter(col('AMOUNT') > 0).join(income_ids, on='ID', how='inner')

    # Count occurrences of each AMOUNT per customer
    amount_counts = positive_txns.groupBy('ID', 'AMOUNT').agg(count('*').alias('amount_count'))

    # Find the mode (AMOUNT with the highest count) and its frequency for each customer
    mode_info = amount_counts.groupBy('ID').agg(
        max(struct(col('amount_count'), col('AMOUNT'))).alias('max_struct')
    ).select(
        col('ID'),
        col('max_struct.AMOUNT').alias('mode_AMOUNT'),
        col('max_struct.amount_count').alias('mode_count')
    )

    # Compute distinct AMOUNT count and overall count(*) per customer
    summary_counts = positive_txns.groupBy('ID').agg(
        countDistinct(col('AMOUNT')).alias('distinct_AMOUNT_count'),
        count('*').alias('total_count')
    )

    # Join mode info with summary counts for comparison
    return mode_info.join(summary_counts, on='ID', how='inner')

mode_frequency_df = calculate_mode_frequency(additional_income, transactions_df)
display(mode_frequency_df)

# If mode count > distinct amount count then assume the mode as the main monthly income and the other
# case was an additional income for that month. If the mode count <= distinct count, then there is not
# enough data points to make the assumption that the mode is the actual or best reflection of the monthly
# income. Thus handle it differently.

# Customers where the mode is the consistent main monthly income
mode_is_main_income = mode_frequency_df.filter(col('mode_count') >= col('distinct_AMOUNT_count')) \
    .select('ID', 'mode_AMOUNT') \
    .withColumnRenamed('mode_AMOUNT', 'INCOME')
display(mode_is_main_income)

# For the remaining customers (mode_count < distinct_AMOUNT_count), there are not enough
# data points to assume the mode is the main monthly income. Use the average positive
# transaction amount instead.
avg_is_main_income = mode_frequency_df.filter(col('mode_count') < col('distinct_AMOUNT_count')) \
    .join(monthly_income_to_inpute_df.groupBy('ID').agg(avg(col('AMOUNT')).alias('INCOME')), on='ID', how='inner') \
    .select('ID', 'INCOME')
display(avg_is_main_income)






In [0]:
# Invetsigate any outlier or unrealistic ages. Being over a 100 is very unlikely
unrealistic_age = customers_df.filter(col('AGE') >= 100)
display(unrealistic_age)

income_value_of_unrealistic_age = unrealistic_age.select('INCOME')

# Extract the actual income value(s) of the customer(s) with unrealistic age
unrealistic_income_values = [row['INCOME'] for row in income_value_of_unrealistic_age.collect() if row['INCOME'] is not None]

# There is one outlier that is female. Plot on histogran to get the INCOME bracket of the customer with the unrealsitic age
female = customers_df.filter(col('GENDER') == 'F')

# Histogram of INCOME for female customers
female_income_pd = female.select('INCOME').toPandas()

import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 6))

income_data = female_income_pd['INCOME'].dropna()

# Determine the ideal bin size using common statistical rules
import math

n_inc = len(income_data)
data_range_inc = income_data.max() - income_data.min()
iqr_inc = np.percentile(income_data, 75) - np.percentile(income_data, 25)
std_inc = income_data.std()

# Sturges' Rule: bins = ceil(log2(n) + 1)
sturges_bins_inc = math.ceil(math.log2(n_inc) + 1) if n_inc > 0 else 0

# Scott's Rule: bin_width = 3.49 * std / n^(1/3); bins = range / bin_width
scotts_width_inc = 3.49 * std_inc / (n_inc ** (1/3)) if n_inc > 0 else 0
scotts_bins_inc = math.ceil(data_range_inc / scotts_width_inc) if scotts_width_inc > 0 else 0

# Freedman-Diaconis Rule: bin_width = 2 * IQR / n^(1/3); bins = range / bin_width
fd_width_inc = 2 * iqr_inc / (n_inc ** (1/3)) if n_inc > 0 else 0
fd_bins_inc = math.ceil(data_range_inc / fd_width_inc) if fd_width_inc > 0 else 0

print(f"--- Ideal Bin Size Analysis for INCOME (n={n_inc}) ---")
print(f"Data range: {data_range_inc:.2f}  | IQR: {iqr_inc:.2f}  | Std: {std_inc:.2f}")
print(f"Sturges' Rule:          {sturges_bins_inc} bins (bin width = {data_range_inc / sturges_bins_inc:.2f})" if sturges_bins_inc else "Sturges' Rule:          N/A")
print(f"Scott's Rule:           {scotts_bins_inc} bins (bin width = {scotts_width_inc:.2f})" if scotts_bins_inc else "Scott's Rule:           N/A")
print(f"Freedman-Diaconis Rule: {fd_bins_inc} bins (bin width = {fd_width_inc:.2f})" if fd_bins_inc else "Freedman-Diaconis Rule: N/A")
print(f"Square-root Rule:       {math.ceil(math.sqrt(n_inc))} bins" if n_inc > 0 else "Square-root Rule:       N/A")

# Use the Freedman-Diaconis bin count (robust to outliers); fall back to Sturges or 30
ideal_bins_inc = fd_bins_inc if fd_bins_inc > 0 else (sturges_bins_inc if sturges_bins_inc > 0 else 30)

n, bins, patches = ax.hist(income_data, bins=ideal_bins_inc, edgecolor='black')

# Highlight the bar(s) in which the unrealistic-age income value(s) fall
for val in unrealistic_income_values:
    for i in range(len(patches)):
        if bins[i] <= val < bins[i + 1]:
            patches[i].set_facecolor('red')
            break

# Draw a vertical line at the unrealistic-age income value for extra clarity
for val in unrealistic_income_values:
    ax.axvline(val, color='red', linestyle='--', linewidth=2, label=f'Unrealistic age INCOME: {val}')

ax.set_xlabel('INCOME')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of INCOME for Female Customers (red bar = unrealistic-age customer)')
ax.legend()
plt.tight_layout()
plt.show()


# Determine the income bracket for the unrealistic age customer
if unrealistic_income_values:
    for val in unrealistic_income_values:
        for i in range(len(bins) - 1):
            if bins[i] <= val < bins[i + 1]:
                print(f"Unrealistic-age INCOME = {val} falls in bracket [{bins[i]:.2f}, {bins[i + 1]:.2f})")
                break
        else:
            # Edge case: value equals the last bin edge
            if val == bins[-1]:
                print(f"Unrealistic-age INCOME = {val} falls in the last bracket [{bins[-2]:.2f}, {bins[-1]:.2f}]")
            else:
                print(f"Unrealistic-age INCOME = {val} is outside the histogram range")
else:
    print("No unrealistic-age income values found.")

# Filter customers_df to INCOME in the same bracket as the unrealistic-age customer's income,
# then plot a histogram of the AGE column for those filtered customers
if unrealistic_income_values:
    for val in unrealistic_income_values:
        # Determine the bin range from the histogram above
        for i in range(len(bins) - 1):
            if bins[i] <= val < bins[i + 1]:
                income_low, income_high = bins[i], bins[i + 1]
                break
        else:
            if val == bins[-1]:
                income_low, income_high = bins[-2], bins[-1]
            else:
                income_low, income_high = val - 5000, val + 5000

        filtered_df = customers_df.filter((col('INCOME') >= income_low) & (col('INCOME') < income_high) & (col('AGE') != 2147483647))
        distinct_age_count = filtered_df.groupBy('AGE').count()
        display(distinct_age_count)

        # Histogram of AGE for customers in the same income bracket as the unrealistic-age customer
        filtered_age_pd = filtered_df.select('AGE').toPandas()

        age_data = filtered_age_pd['AGE'].dropna()

        # Determine the ideal bin size using common statistical rules
        import math

        n = len(age_data)
        data_range = age_data.max() - age_data.min()
        iqr = np.percentile(age_data, 75) - np.percentile(age_data, 25)
        std = age_data.std()

        # Sturges' Rule: bins = ceil(log2(n) + 1)
        sturges_bins = math.ceil(math.log2(n) + 1) if n > 0 else 0

        # Scott's Rule: bin_width = 3.49 * std / n^(1/3); bins = range / bin_width
        scotts_width = 3.49 * std / (n ** (1/3)) if n > 0 else 0
        scotts_bins = math.ceil(data_range / scotts_width) if scotts_width > 0 else 0

        # Freedman-Diaconis Rule: bin_width = 2 * IQR / n^(1/3); bins = range / bin_width
        fd_width = 2 * iqr / (n ** (1/3)) if n > 0 else 0
        fd_bins = math.ceil(data_range / fd_width) if fd_width > 0 else 0

        print(f"--- Ideal Bin Size Analysis for AGE (n={n}) ---")
        print(f"Data range: {data_range:.2f}  | IQR: {iqr:.2f}  | Std: {std:.2f}")
        print(f"Sturges' Rule:          {sturges_bins} bins (bin width = {data_range / sturges_bins:.2f})" if sturges_bins else "Sturges' Rule:          N/A")
        print(f"Scott's Rule:           {scotts_bins} bins (bin width = {scotts_width:.2f})" if scotts_bins else "Scott's Rule:           N/A")
        print(f"Freedman-Diaconis Rule: {fd_bins} bins (bin width = {fd_width:.2f})" if fd_bins else "Freedman-Diaconis Rule: N/A")
        print(f"Square-root Rule:       {math.ceil(math.sqrt(n))} bins" if n > 0 else "Square-root Rule:       N/A")

        # Plot the histogram using the Freedman-Diaconis bin count (robust to outliers)
        ideal_bins = fd_bins if fd_bins > 0 else (sturges_bins if sturges_bins > 0 else 30)

        fig2, ax2 = plt.subplots(figsize=(10, 6))
        ax2.hist(age_data, bins=ideal_bins, edgecolor='black')
        ax2.set_xlabel('AGE')
        ax2.set_ylabel('Frequency')
        ax2.set_title(f'Distribution of AGE for Customers with INCOME in [{income_low:.2f}, {income_high:.2f})\n(Using {ideal_bins} bins from Freedman-Diaconis rule)')
        plt.tight_layout()
        plt.show()

        # Calculate mean, mode, and median of the AGE data
        from scipy import stats as scipy_stats

        mean_age = age_data.mean()
        median_age = age_data.median()
        mode_result = scipy_stats.mode(age_data, keepdims=False)
        mode_age = mode_result.mode

        print(f"--- AGE Statistics (INCOME bracket [{income_low:.2f}, {income_high:.2f})) ---")
        print(f"Mean:   {mean_age:.2f}")
        print(f"Median: {median_age:.2f}")
        print(f"Mode:   {mode_age} (count: {mode_result.count})")

# It looks like the age distribution is symmetrical and is slighly positively skewed. Best value to infer the age is the median as it is robust to skewness, outliers, it is the best central estimate when mean ≈ median and thus the most stable imputation value.
        # Impute the unrealistic age with the median age from the same income bracket
        customers_df = customers_df.withColumn(
            'AGE',
            when((col('AGE') >= 100) & (col('INCOME') == val), float(median_age)).otherwise(col('AGE'))
        )
        print(f"Imputed AGE = {median_age:.2f} for customer(s) with INCOME = {val} and unrealistic age")

In [0]:
# Get the customer ID's that have missing income data. There are over 1,324 customers with missing income. 
monthly_income_inputed_df = customers_df.filter(col('INCOME').isNull()).select('ID').join(transactions_df.filter(col('AMOUNT') > 0), on = 'ID', how = 'left').groupby("ID").agg(mode(col('AMOUNT')).alias('AMOUNT'))
#display(monthly_income_inputed_df)

# Impute missing income data using the Mode of the monthly income data
customers_df = customers_df.join(monthly_income_inputed_df, on='ID', how='left').withColumn('INCOME', when(col('INCOME').isNull(), col('AMOUNT')).otherwise(col('INCOME'))).drop('AMOUNT')
display(customers_df)


# Transactional Data
- Clean up the missing balance data

In [0]:
display(transactions_df)

Databricks filtered table. Run in Databricks to view.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when, last, first, sum as spark_sum, count

# Window partitioned by ID, ordered by __index_level_0__ ascending
w = Window.partitionBy('ID').orderBy('__index_level_0__').rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Get last non-null BALANCE and create group boundaries (new group starts at each non-null BALANCE)
null_balance_transactions_df = transactions_df \
    .withColumn('last_balance', last(col('BALANCE'), ignorenulls=True).over(w)) \
    .withColumn('group_id', count(when(col('BALANCE').isNotNull(), 1)).over(w))

# Window within each group (consecutive rows sharing the same last non-null BALANCE)
w_group = Window.partitionBy('ID', 'group_id').orderBy('__index_level_0__').rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Cumulative sum of AMOUNT within group, and first AMOUNT in group (the non-null BALANCE row)
null_balance_transactions_df = null_balance_transactions_df \
    .withColumn('cumsum_amount', spark_sum(col('AMOUNT')).over(w_group)) \
    .withColumn('first_amount', first(col('AMOUNT')).over(w_group))

# Forward-looking window: get the next non-null BALANCE for backward fill of leading nulls
# (rows where BALANCE is null and there is no preceding non-null BALANCE to forward-fill from)
w_forward = Window.partitionBy('ID').orderBy('__index_level_0__').rowsBetween(Window.currentRow, Window.unboundedFollowing)
null_balance_transactions_df = null_balance_transactions_df \
    .withColumn('next_balance', first(col('BALANCE'), ignorenulls=True).over(w_forward))

# Window from current row to end within group — used for backward fill of leading nulls
w_group_after = Window.partitionBy('ID', 'group_id').orderBy('__index_level_0__').rowsBetween(Window.currentRow, Window.unboundedFollowing)
null_balance_transactions_df = null_balance_transactions_df \
    .withColumn('sum_after_in_group', spark_sum(col('AMOUNT')).over(w_group_after))

# Impute null BALANCE:
# - Leading nulls (last_balance is null, i.e. no preceding non-null BALANCE): backward fill.
#   Use the next non-null BALANCE and subtract the sum of AMOUNTs from the current row to the
#   end of the leading-null group (group_id = 0).
#   E.g., rows 1,2 are null and row 3 has BALANCE:
#     row2.BALANCE = row3.BALANCE - row2.AMOUNT
#     row1.BALANCE = row3.BALANCE - row1.AMOUNT - row2.AMOUNT
# - Trailing nulls (last_balance is not null): forward fill.
#   imputed = last_balance + cumsum_amount - first_amount
null_balance_transactions_df = null_balance_transactions_df \
    .withColumn('BALANCE',
        when(col('BALANCE').isNull(),
            when(col('last_balance').isNull(),
                col('next_balance') - col('sum_after_in_group')
            ).otherwise(
                col('last_balance') + col('cumsum_amount') - col('first_amount')
            )
        ).otherwise(col('BALANCE'))
    ) \
    .drop('last_balance', 'group_id', 'cumsum_amount', 'first_amount',
          'next_balance', 'sum_after_in_group')

display(null_balance_transactions_df.orderBy('ID', '__index_level_0__'))

Databricks visualization. Run in Databricks to view.

Databricks filtered table. Run in Databricks to view.

In [0]:
%skip
from pyspark.sql.functions import col

transactions_clean_df = null_balance_transactions_df

output_path = f"{ROOT}/data/02_intermediate/transactions_prep"

transactions_clean_df.write.mode("overwrite").format("delta").save(output_path)

print(f"Saved transactions_clean Delta table to {output_path}")
print(f"Row count: {transactions_clean_df.count()}")

In [0]:
%skip
from pyspark.sql.functions import col

customers_clean_df = customers_df

output_path = f"{ROOT}/data/02_intermediate/customers_prep"

customers_clean_df.write.mode("overwrite").format("delta").save(output_path)

print(f"Saved customers_clean Delta table to {output_path}")
print(f"Row count: {customers_clean_df.count()}")

In [0]:
import sys
ROOT = "/Workspace/Users/melaniepieterse1@gmail.com/Loan_Offer_Model"
CATALOG_PATH = f"{ROOT}/conf/base/catalog.yml"
sys.path.insert(0, ROOT)

from src.pipelines.data_processing.pipeline import run_pipeline

run_pipeline(spark, CATALOG_PATH)